# RPA_CCQE_UBGenie knob grid offset

The 7-point `RPA_CCQE_UBGenie` vector stored in the overlay files is not centred on the tuned CV:
for every event the knob touches, the entry equal to `TunedCentralValue_UBGenie` sits at knob
+0.62, not at knob 0. PROfit's `force_0_cv` takes knob 0 as the CV, so the spline is read 0.62 of
a grid step off. `MaCCQE_UBGenie` is shown next to it as the reference for how the vectors are
supposed to look. Notebook 19 (Section 6) found the symptom as a 31% match at every index; this
notebook shows the per-event vectors behind it.

Events come from the selected-event cache written by notebook 19
(`tables/injection_study/selected_events.npz`): selected 1mu1p overlay events, no data, EXT,
dirt or NuWro.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

start = Path.cwd().resolve()
REPO = next(
    (parent / 'ma_zexp' for parent in (start, *start.parents)
     if (parent / 'ma_zexp' / 'tables').is_dir()),
    None,
)
if REPO is None:
    raise FileNotFoundError('Could not locate ma_zexp')

CACHE = REPO / 'tables' / 'injection_study' / 'selected_events.npz'
FIG_DIR = REPO / 'figs' / 'knob_checks'
FIG_DIR.mkdir(parents=True, exist_ok=True)

KNOB = 'RPA_CCQE_UBGenie'
REFERENCE = 'MaCCQE_UBGenie'
KNOTS = np.arange(-3, 4)
N_DRAW = 300       # events drawn in the per-event plots
SEED = 1

## Load the cache and get the numbers

`T` is the tuned CV weight. An event is *affected* by a knob if its vector is not constant and
equal to `T`. For the affected events the RPA vector is linear in the knob index (GENIE clips it
at zero), so the knob value where it equals `T` follows from the two entries around zero.

In [ ]:
with np.load(CACHE) as f:
    T = f['TunedCentralValue_UBGenie'][:, 0]
    V = {name: f[name] for name in (KNOB, REFERENCE)}
    SIGNAL = f['afro_1mu1p_true'] == 1

good = (T > 0) & np.isfinite(T)
flat = {name: np.all(np.isclose(v, T[:, None], rtol=1e-6), axis=1) for name, v in V.items()}
affected = {name: good & ~flat[name] for name in V}
ratio = {name: V[name][affected[name]] / T[affected[name], None] for name in V}

r = ratio[KNOB]
slope = r[:, 4] - r[:, 3]
k_cross = (1 - r[:, 3]) / slope
k_cross = k_cross[np.isfinite(k_cross)]
K_STAR = float(np.median(k_cross))

print(f'{len(T)} selected events, {good.sum()} with T > 0')
for name in V:
    print(f'{name}: {affected[name].sum()} affected, {flat[name].sum()} with a flat vector ({flat[name].mean():.1%})')
print(f'{KNOB}: vector equals T at knob {K_STAR:+.4f} '
      f'(spread over events: {np.quantile(k_cross, 0.01):+.4f} to {np.quantile(k_cross, 0.99):+.4f}, 1st to 99th percentile)')
print(f'{KNOB}: entries clipped to exactly 0 in {(V[KNOB][affected[KNOB]] == 0).any(axis=1).sum()} affected events')
print(f'{REFERENCE}: knob 0 equals T for every event: {np.allclose(V[REFERENCE][good, 3], T[good])}')

## RPA vectors event by event

Each line is one affected event, the stored entry divided by `T`. The lines all cross 1 at the
same knob value, marked by the vertical line, and it is not zero.

In [ ]:
rng = np.random.default_rng(SEED)
draw = rng.choice(np.where(affected[KNOB])[0], N_DRAW, replace=False)

fig, ax = plt.subplots(figsize=(5.5, 4))
for i in draw:
    ax.plot(KNOTS, V[KNOB][i] / T[i], color='tab:blue', lw=0.5, alpha=0.12)
ax.plot(KNOTS, np.median(ratio[KNOB], axis=0), color='tab:blue', lw=2, label='median of affected events')
ax.axhline(1, color='k', lw=0.8)
ax.axvline(0, color='grey', lw=0.8, ls='--', label='knob 0 (PROfit CV)')
ax.axvline(K_STAR, color='tab:red', lw=1.2, label=f'vector = T at knob {K_STAR:+.2f}')
ax.set_xlabel('knob index')
ax.set_ylabel('stored weight / T')
ax.set_xticks(KNOTS)
ax.set_ylim(0.3, 1.9)
ax.set_title(f'{KNOB}, {N_DRAW} of {affected[KNOB].sum()} affected events')
ax.legend(loc='upper right', fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / 'rpa_vectors_per_event.pdf')
plt.show()

## MaCCQE for comparison

Same events, same axes. Here knob 0 is the tuned CV and every line goes through (0, 1).

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
for i in draw:
    ax.plot(KNOTS, V[REFERENCE][i] / T[i], color='tab:orange', lw=0.5, alpha=0.12)
ax.plot(KNOTS, np.median(ratio[REFERENCE], axis=0), color='tab:orange', lw=2, label='median of affected events')
ax.axhline(1, color='k', lw=0.8)
ax.axvline(0, color='grey', lw=0.8, ls='--', label='knob 0 (PROfit CV)')
ax.set_xlabel('knob index')
ax.set_ylabel('stored weight / T')
ax.set_xticks(KNOTS)
ax.set_ylim(0.3, 1.9)
ax.set_title(f'{REFERENCE}, same {N_DRAW} events')
ax.legend(loc='upper left', fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / 'maccqe_vectors_per_event.pdf')
plt.show()

## Which entry equals the tune weight

The check of notebook 19 Section 6 as a plot: the fraction of selected events whose entry at
each knob equals `T`. For MaCCQE it is every event at knob 0. For RPA it is the same 31% at every
knob, the events the knob does not touch, and no entry matches for the rest.

In [ ]:
match = {name: np.array([np.isclose(v[:, j], T, rtol=1e-5).mean() for j in range(len(KNOTS))]) for name, v in V.items()}

fig, ax = plt.subplots(figsize=(5.5, 3.6))
w = 0.35
ax.bar(KNOTS - w / 2, match[KNOB], width=w, color='tab:blue', label=KNOB)
ax.bar(KNOTS + w / 2, match[REFERENCE], width=w, color='tab:orange', label=REFERENCE)
ax.axhline(flat[KNOB].mean(), color='grey', lw=0.8, ls='--')
ax.text(3.4, flat[KNOB].mean() + 0.02, f'{flat[KNOB].mean():.1%} untouched', ha='right', fontsize=8, color='dimgrey')
ax.set_xlabel('knob index')
ax.set_ylabel('fraction of events with entry = T')
ax.set_xticks(KNOTS)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / 'tune_weight_index_match.pdf')
plt.show()